In [1]:
# Install all required libraries for image processing, deep learning, 
# feature extraction and ensemble machine learning models.
!pip install torch torchvision xgboost lightgbm catboost

In [2]:
# Import required libraries
import os 
from pathlib import Path 
from typing import Tuple, List, Dict

import numpy as np
import pandas as pd
from PIL import Image 

from sklearn.base import clone 
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier 

import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader 

from torchvision import models, transforms

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
data_dir = Path(".")
output_path = Path("task2_predictions.csv")
RANDOM_STATE = 42
print (os.listdir(data_dir))

In [ ]:
# Load train/test metadata and merge the provided feature files.
def load_csv_features(data_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    train_meta = pd.read_csv(data_dir / "train_metadata.csv")
    test_meta = pd.read_csv(data_dir / "test_metadata.csv")
    class_mapping = pd.read_csv(data_dir / "class_mapping.csv")

    all_meta = pd.concat(
        [
            train_meta[["image_id", "image_path", "class_id", "class_name"]],
            test_meta.assign(class_id=np.nan, class_name=np.nan)[
                ["image_id", "image_path", "class_id", "class_name"]
            ],
        ],
        ignore_index=True,
    )

    features = all_meta.copy()

    for filename in [
        "color_histogram.csv",
        "hog_pca.csv",
        "additional_features.csv"
    ]:
        feature_df = pd.read_csv(data_dir / filename)
        features = features.merge(feature_df, on="image_id", how="left")

    return features, train_meta, class_mapping

In [ ]:
# extract extra image features which is useful for fine-grained bird classification
def engineer_image_features(data_dir: Path, metadata: pd.DataFrame) -> pd.DataFrame:

    rows: List[Dict[str, float]] = []

    for _, row in metadata.iterrows():

        image_path = data_dir / row["image_path"]
        image = Image.open(image_path).convert("RGB")

        arr = np.asarray(image).astype(np.float32) / 255.0

        r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
        gray = 0.299 * r + 0.587 * g + 0.114 * b

        image_features = {"image_id": row["image_id"]}

        # RGB statistics
        for name, channel in zip(["r", "g", "b"], [r, g, b]):
            image_features[f"{name}_mean"] = float(channel.mean())
            image_features[f"{name}_std"] = float(channel.std())
            image_features[f"{name}_q25"] = float(np.quantile(channel, 0.25))
            image_features[f"{name}_q50"] = float(np.quantile(channel, 0.50))
            image_features[f"{name}_q75"] = float(np.quantile(channel, 0.75))

        # Colour-opponent features
        rg = r - g
        yb = 0.5 * (r + g) - b

        image_features["rg_mean_abs"] = float(np.mean(np.abs(rg)))
        image_features["yb_mean_abs"] = float(np.mean(np.abs(yb)))
        image_features["rg_std"] = float(np.std(rg))
        image_features["yb_std"] = float(np.std(yb))

        # Brightness features
        image_features["gray_mean"] = float(gray.mean())
        image_features["gray_std"] = float(gray.std())

        # Fine-grained birds often differ in head/wing/body markings,
        # so we use spatial grid features.
        cell = 32

        for i in range(4):
            for j in range(4):
                patch = gray[
                    i * cell:(i + 1) * cell,
                    j * cell:(j + 1) * cell
                ]
                image_features[f"grid_gray_{i}_{j}"] = float(patch.mean())
                image_features[f"grid_gray_std_{i}_{j}"] = float(patch.std())

        # Centre vs border contrast
        centre = gray[32:96, 32:96]

        border_mask = np.ones_like(gray, dtype=bool)
        border_mask[32:96, 32:96] = False
        border = gray[border_mask]

        image_features["centre_mean"] = float(centre.mean())
        image_features["border_mean"] = float(border.mean())
        image_features["centre_border_diff"] = float(centre.mean() - border.mean())

        rows.append(image_features)

    return pd.DataFrame(rows)

In [ ]:
# Configure computation device (MPS/CUDA/CPU),
# load pretrained ResNet18 model and remove the
# final classification layer for feature extraction.

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

resnet = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# remove final classifier
resnet.fc = nn.Identity()

resnet = resnet.to(device)
resnet.eval()

In [ ]:
# Define image preprocessing pipeline for ResNet18, including resizing, tensor conversion and normalisation.
resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Create a custom PyTorch Dataset class to load and preprocess images for batch feature extraction.
class ImageFeatureDataset(Dataset):

    def __init__(self, metadata, data_dir):

        self.metadata = metadata.reset_index(drop=True)
        self.data_dir = Path(data_dir)

    def __len__(self):

        return len(self.metadata)

    def __getitem__(self, idx):

        row = self.metadata.iloc[idx]

        image_path = self.data_dir / row["image_path"]

        image = Image.open(image_path).convert("RGB")

        image = resnet_transform(image)

        return image, row["image_id"]

In [ ]:
# Extract deep image embeddings from ResNet18 using batched inference and return them as feature vectors.
def extract_resnet_features(
    metadata,
    data_dir,
    batch_size=32
):

    dataset = ImageFeatureDataset(
        metadata,
        data_dir
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    all_features = []
    all_ids = []

    with torch.no_grad():

        for images, image_ids in loader:

            images = images.to(device)

            features = resnet(images)

            features = features.cpu().numpy()

            all_features.append(features)

            all_ids.extend(image_ids)

    all_features = np.vstack(all_features)

    feature_df = pd.DataFrame(
        all_features,
        columns=[
            f"resnet_{i}"
            for i in range(all_features.shape[1])
        ]
    )

    feature_df.insert(0, "image_id", all_ids)

    return feature_df

In [ ]:
# Load cached ResNet18 features if available, otherwise extract features from all images and save them for reuse.
train_meta = pd.read_csv(data_dir / "train_metadata.csv")

test_meta = pd.read_csv(data_dir / "test_metadata.csv")

all_meta = pd.concat([train_meta[["image_id", "image_path"]],
                      test_meta[["image_id", "image_path"]]],
                      ignore_index=True)

cache_path = (data_dir / "resnet18_features_task2.csv")

if cache_path.exists():
    resnet_features = pd.read_csv(cache_path)
else:
    resnet_features = extract_resnet_features(all_meta, data_dir)

    resnet_features.to_csv(cache_path,
                           index=False)

In [ ]:
# prepare X, y, X_test for task 2
def prepare_data(data_dir: Path, resnet_features: pd.DataFrame, feature_mode="all"):
    """
    Prepare data using different feature sets.

    feature_mode options:
    - "provided_engineered": provided CSV features + handcrafted image features
    - "resnet_only": ResNet18 features only
    - "all": provided CSV features + engineered features + ResNet18 features
    """

    features, train_meta, class_mapping = load_csv_features(data_dir)

    # cache engineered features
    cache_path = data_dir / "engineered_image_features_task2.csv"

    if cache_path.exists():
        image_features = pd.read_csv(cache_path)
    else:
        image_features = engineer_image_features(
            data_dir,
            features[["image_id", "image_path"]]
        )
        image_features.to_csv(cache_path, index=False)

    metadata_cols = ["image_id", "image_path", "class_id", "class_name"]

    if feature_mode == "provided_engineered":
        features = features.merge(
            image_features,
            on="image_id",
            how="left"
        )

    elif feature_mode == "resnet_only":
        features = features[metadata_cols].merge(
            resnet_features,
            on="image_id",
            how="left"
        )

    elif feature_mode == "all":
        features = features.merge(
            image_features,
            on="image_id",
            how="left"
        )

        features = features.merge(
            resnet_features,
            on="image_id",
            how="left"
        )

    else:
        raise ValueError(
            "feature_mode must be 'provided_engineered', 'resnet_only', or 'all'"
        )

    train_df = features[features["class_id"].notna()].copy()
    test_df = features[features["class_id"].isna()].copy()

    feature_cols = [
        c for c in features.columns
        if c not in ["image_id", "image_path", "class_id", "class_name"]
    ]

    X = train_df[feature_cols].fillna(0.0).values
    y = train_df["class_id"].astype(int).values
    X_test = test_df[feature_cols].fillna(0.0).values

    class_lookup = (
        class_mapping
        .sort_values("class_id")
        .set_index("class_id")["class_name"]
        .to_dict()
    )

    test_ids = test_df[["image_id"]].reset_index(drop=True)

    return X, y, X_test, test_ids, class_lookup

In [ ]:
# Build multiple machine learning models and ensemble methods for comparison on the bird classification task.
def build_models() -> Dict[str, object]:
    models = {}

    # Extra Trees: performed well with ResNet features
    models["extra_trees"] = ExtraTreesClassifier(
        n_estimators=500,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        n_jobs=1,
        random_state=RANDOM_STATE
    )

    # LightGBM
    models["lightgbm"] = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.04,
        num_leaves=15,
        min_child_samples=10,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        verbose=-1
    )

    # catboost
    models["catboost"] = CatBoostClassifier(
        iterations=250,
        learning_rate=0.04,
        depth=5,
        loss_function="MultiClass",
        eval_metric="Accuracy",
        random_state=RANDOM_STATE,
        verbose=0
    )

    # Soft voting ensemble
    models["ensemble_et_cat_lgbm"] = VotingClassifier(
    estimators=[
        ("et", clone(models["extra_trees"])),
        ("cat", clone(models["catboost"])),
        ("lgbm", clone(models["lightgbm"]))
    ],
    voting="soft",
    weights=[3, 2, 2],
    n_jobs=1
)

    return models